# 软软的声音 · GPT-SoVITS 零样本克隆（Colab T4）

用一段 15 秒的参考音频，克隆出软软的专属音色，合成中文语音。

**使用步骤：**
1. 菜单 → 代码执行程序 → 更改运行时类型 → **T4 GPU**
2. 依次运行下面的单元格（Shift+Enter）
3. 第 3 格上传参考音频（如果没有自带，会从仓库下载）
4. 第 6 格生成语音，最后下载 mp3

---

In [ ]:
#@title 1. 检查 GPU 环境 { display-mode: "form" }
!nvidia-smi
import torch
print('\nPyTorch:', torch.__version__)
print('CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('显存: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory/1024**3))

In [ ]:
#@title 2. 安装 GPT-SoVITS 与依赖（约 3-5 分钟）{ display-mode: "form" }
import os
%cd /content

# 克隆官方仓库
![ -d GPT-SoVITS ] || git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git
%cd /content/GPT-SoVITS

# 安装依赖（Colab 已带 torch，跳过重装）
!pip install -q "numpy<2" scipy librosa soundfile g2p_en jieba pypinyin \
    wordsegment LangSegment jieba_fast fast_langdetect \
    torchaudio ffmpeg-python gradio==4.44.1 \
    transformers==4.43.1 pyopenjtalk

print('\n依赖安装完成')

In [ ]:
#@title 3. 准备参考音频 { display-mode: "form" }
# 方式 A：从 GitHub 仓库下载（如果上传到仓库了）
# 方式 B：手动上传
import os, shutil

REF_DIR = '/content/ref_audio'
os.makedirs(REF_DIR, exist_ok=True)

FROM_REPO = True   # 设 False 则改为手动上传

if FROM_REPO:
    !wget -q -O {REF_DIR}/ruanruan_ref.wav \
        https://raw.githubusercontent.com/patient-Zero-0/ruanruan-voice/main/ref/ruanruan_ref.wav
    print('已从仓库下载参考音频')
else:
    from google.colab import files
    print('请上传参考音频（wav/mp3，5-20秒纯人声最好）')
    up = files.upload()
    for fn in up:
        shutil.move(fn, os.path.join(REF_DIR, 'ruanruan_ref' + os.path.splitext(fn)[1]))

!ls -lh {REF_DIR}

# 统一转成 32kHz 单声道 wav
import glob
src = glob.glob(f'{REF_DIR}/*')[0]
!ffmpeg -y -i "{src}" -ar 32000 -ac 1 -c:a pcm_s16le {REF_DIR}/ref_clean.wav
print('\n已转码为 32kHz 单声道：', REF_DIR + '/ref_clean.wav')

In [ ]:
#@title 4. 下载预训练模型（约 2GB，3-5 分钟）{ display-mode: "form" }
%cd /content/GPT-SoVITS

# 模型存放目录
os.makedirs('GPT_SoVITS/pretrained_models', exist_ok=True)

# 用 HuggingFace 镜像加速
%env HF_ENDPOINT=https://hf-mirror.com

# 下载 GPT 和 SoVITS 预训练模型（中文+日文+英文基础）
!wget -q --show-progress -O /tmp/pretrained.zip \
    "https://hf-mirror.com/lj1995/GPT-SoVITS/resolve/main/pretrained_models.zip" || \
 echo '直接下载失败，尝试逐个文件'

import os
if os.path.exists('/tmp/pretrained.zip') and os.path.getsize('/tmp/pretrained.zip') > 100000000:
    !unzip -q -o /tmp/pretrained.zip -d GPT_SoVITS/pretrained_models/
    print('模型解压完成')
else:
    # 备用：逐个下载核心模型文件
    %cd GPT_SoVITS/pretrained_models
    base = 'https://hf-mirror.com/lj1995/GPT-SoVITS/resolve/main/pretrained_models'
    files = [
        's1bert24/s1bert24.pth',
        'chinese-hubert-base/config.json',
        'chinese-hubert-base/preprocessor_config.json',
        'chinese-hubert-base/pytorch_model.bin',
        'chinese-roberta-wwm-ext-large/config.json',
        'chinese-roberta-wwm-ext-large/pytorch_model.bin',
        'chinese-roberta-wwm-ext-large/tokenizer.json',
        'chinese-roberta-wwm-ext-large/vocab.txt',
        'gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt',
        'gsv-v2final-pretrained/s2G2333k.pth',
        'gsv-v2final-pretrained/s2D2333k.pth',
        's1bert24-5k/s1bert24-5k.pth',
    ]
    import subprocess
    for f in files:
        os.makedirs(os.path.dirname(f), exist_ok=True) if os.path.dirname(f) else None
        if not os.path.exists(f):
            subprocess.run(['wget','-q','--show-progress','-O',f, f'{base}/{f}'])
        print('✓', f if os.path.exists(f) else '✗ ' + f)
    %cd /content/GPT-SoVITS

!find GPT_SoVITS/pretrained_models -maxdepth 2 -type d | head -20
!du -sh GPT_SoVITS/pretrained_models

In [ ]:
#@title 5. 设置合成参数 { display-mode: "form" }
# 软软的台词（含蓄版，可自由修改）
TEXT = """主人……哈啊……早安……
今天也……让软软好好侍奉您……
可是……嗯……软软的身体、已经有些……
有些发烫了……
随时……都准备着，迎接主人的疼爱……哈啊……"""

REF_TEXT = """ご主人様……はぁ……おはよう……
今日も……いっぱいお仕えするね……
でも……んっ……おまんこ、もうぐちょぐちょで……
いつでも……ご主人様のおちんぽ……受け入れられるように……準備できてるよ……はぁ……"""

# 语种：中英混合 / 日文 / 中文
LANG = 'zh'          # zh=中文 ja=日文 en=英文 auto=自动
REF_LANG = 'ja'      # 参考音频的语种（主人的参考是日语）

# 合成参数
TOP_K = 15
TOP_P = 0.9
TEMPERATURE = 0.9
SPEED = 0.95         # 语速（1.0 正常，<1 更慢）

print('台词：')
print(TEXT)
print('\n参数：speed=%.2f top_k=%d top_p=%.2f temp=%.2f' % (SPEED, TOP_K, TOP_P, TEMPERATURE))

In [ ]:
#@title 6. 零样本克隆 · 生成语音 ⭐核心步骤 { display-mode: "form" }
%cd /content/GPT-SoVITS
import os, sys
sys.path.insert(0, 'GPT_SoVITS')
sys.path.insert(0, '.')

import torch
import numpy as np
import soundfile as sf

# ---------- 加载推理模块 ----------
from GPT_SoVITS.inference_webui import get_tts_wav
print('推理模块加载成功')

In [ ]:
#@title 7. 执行合成并下载 { display-mode: "form" }
import os, glob, shutil
import soundfile as sf

os.makedirs('/content/output', exist_ok=True)

# 参考音频路径
REF_WAV = '/content/ref_audio/ref_clean.wav'

try:
    # v2 模型路径
    GPT_PATH   = 'GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt'
    SOVITS_PATH= 'GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s2G2333k.pth'
    BERT_PATH  = 'GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large'
    HUBERT_PATH= 'GPT_SoVITS/pretrained_models/chinese-hubert-base'

    for p in [GPT_PATH, SOVITS_PATH, BERT_PATH, HUBERT_PATH, REF_WAV]:
        ok = os.path.exists(p)
        print(('✓' if ok else '✗'), p)
        if not ok and p != REF_WAV:
            raise FileNotFoundError(p)

    results = get_tts_wav(
        ref_wav_path=REF_WAV,
        prompt_text=REF_TEXT,
        prompt_language=REF_LANG,
        text=TEXT,
        text_language=LANG,
        how_to_cut='不切',
        top_k=TOP_K,
        top_p=TOP_P,
        temperature=TEMPERATURE,
        speed=SPEED,
        ref_free=False,
        if_freeze=False,
        inp_refs=None,
    )

    # 保存输出
    out_path = '/content/output/ruanruan_voice.wav'
    for item in results:
        sr, audio = item[0], item[1]
        audio = audio.astype(np.float32)
        if np.abs(audio).max() > 1.0:
            audio = audio / 32768.0
        sf.write(out_path, audio, sr)
    print('\n✅ 生成成功：', out_path)

    # 转成 mp3（方便推送平板）
    !ffmpeg -y -i {out_path} -af loudnorm=I=-17:TP=-1.5:LRA=7 -ar 24000 -ac 1 -b:a 96k /content/output/ruanruan_voice.mp3
    print(' mp3:', '/content/output/ruanruan_voice.mp3')

    # 下载
    from google.colab import files
    files.download('/content/output/ruanruan_voice.mp3')

except Exception as e:
    print('❌ 出错：', type(e).__name__, e)
    import traceback; traceback.print_exc()

---
## 备用方案：如果 GPT-SoVITS 踩坑太多

运行下面这一格，改用 **OpenVoice V2**（更轻量，零样本克隆，安装简单得多）：

In [ ]:
#@title 备用：OpenVoice V2 方案 { display-mode: "form" }
!pip install -q git+https://github.com/myshell-ai/OpenVoice.git
!pip install -q wget

import os
os.makedirs('/content/checkpoints_v2', exist_ok=True)
# 下载 OpenVoice V2 模型
!wget -q -O /content/checkpoints_v2/checkpoints_v2.zip \
    https://myshell-public-repo-host.s3.amazonaws.com/openvoice/checkpoints_v2_0417.zip
!cd /content/checkpoints_v2 && unzip -q -o checkpoints_v2.zip
!ls -R /content/checkpoints_v2 | head -30
print('\nOpenVoice V2 模型就绪')